# Inside a small CNN: two datasets, eight optimizers

This notebook is the training code behind the [CNN Optimizer Explorer](../dashboard/) dashboard. It works on either of two datasets, chosen with one setting:

- **CIFAR-10**: 32 × 32 color photos of 10 kinds of objects (airplanes, cats, trucks, ...).
- **Fashion-MNIST**: 28 × 28 grayscale pictures of 10 kinds of clothing (shirts, sneakers, bags, ...).

For the chosen dataset it:

1. loads and explores the data,
2. builds an MLP baseline and a small CNN, and checks the CNN's parameter count by hand,
3. trains both and evaluates the CNN (learning curves, confusion matrix, misclassified images),
4. trains the same CNN with eight optimizers and compares them,
5. saves the models in the format the dashboard's export script reads.

Every explanation has an Indonesian translation you can expand. Run the notebook top to bottom, then change `DATASET` and run it again for the other dataset. On a Colab GPU (*Runtime → Change runtime type → T4 GPU*) one full run takes about 10 minutes.

<details>
<summary>Bahasa Indonesia</summary>

# Di dalam CNN kecil: dua dataset, delapan optimizer

Notebook ini adalah kode training di balik dashboard [CNN Optimizer Explorer](../dashboard/). Notebook ini bisa dipakai untuk salah satu dari dua dataset, cukup dengan satu pengaturan:

- **CIFAR-10**: foto berwarna 32 × 32 dari 10 jenis objek (pesawat, kucing, truk, ...).
- **Fashion-MNIST**: gambar grayscale 28 × 28 dari 10 jenis pakaian (kemeja, sepatu kets, tas, ...).

Untuk dataset yang dipilih, notebook ini:

1. memuat dan mengeksplorasi data,
2. membuat baseline MLP dan CNN kecil, lalu mencocokkan jumlah parameter CNN dengan hitungan manual,
3. melatih keduanya dan mengevaluasi CNN (kurva training, confusion matrix, citra yang salah klasifikasi),
4. melatih CNN yang sama dengan delapan optimizer lalu membandingkannya,
5. menyimpan model dalam format yang dibaca skrip ekspor dashboard.

Setiap penjelasan punya terjemahan bahasa Indonesia yang bisa dibuka seperti ini. Jalankan notebook dari atas ke bawah, lalu ganti `DATASET` dan jalankan lagi untuk dataset lainnya. Dengan GPU Colab (*Runtime → Change runtime type → T4 GPU*) satu run penuh butuh sekitar 10 menit.

</details>

## 1. Setup

Choose the dataset here. `EPOCHS` controls every training run, and `QUICK_RUN = True` trains on 5,000 images for one epoch, which is useful to check that everything works before a full run. The fixed seed makes runs repeatable.

<details>
<summary>Bahasa Indonesia</summary>

## 1. Setup

Pilih dataset di sini. `EPOCHS` mengatur semua proses training, dan `QUICK_RUN = True` hanya melatih 5.000 citra selama satu epoch, berguna untuk mengecek semuanya berjalan sebelum run penuh. Seed yang tetap membuat hasil bisa diulang.

</details>

In [ ]:
DATASET = "cifar10"        # "cifar10" or "fashion"
EPOCHS = 10
BATCH_SIZE = 128
QUICK_RUN = False          # True: 5,000 training images and 1 epoch, for a fast check
SEED = 42

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

if QUICK_RUN:
    EPOCHS = 1
keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none, training will run on CPU")

## 2. Load the data

Both datasets come with `keras.datasets` and download on first use:

| | CIFAR-10 | Fashion-MNIST |
|---|---|---|
| Images | 60,000 color, 32 × 32 | 70,000 grayscale, 28 × 28 |
| Split | 50,000 train, 10,000 test | 60,000 train, 10,000 test |
| Download | about 170 MB | about 30 MB |
| Source | [cs.toronto.edu/~kriz/cifar.html](https://www.cs.toronto.edu/~kriz/cifar.html) | [github.com/zalandoresearch/fashion-mnist](https://github.com/zalandoresearch/fashion-mnist) |

Pixel values are scaled from 0–255 to 0–1. Every image gets an explicit channel axis, `(N, H, W, C)`: CIFAR-10 already has 3 channels (RGB), while Fashion-MNIST needs a channel axis of size 1 added.

<details>
<summary>Bahasa Indonesia</summary>

## 2. Memuat data

Kedua dataset tersedia di `keras.datasets` dan diunduh saat pertama kali dipakai:

| | CIFAR-10 | Fashion-MNIST |
|---|---|---|
| Citra | 60.000 berwarna, 32 × 32 | 70.000 grayscale, 28 × 28 |
| Pembagian | 50.000 latih, 10.000 uji | 60.000 latih, 10.000 uji |
| Unduhan | sekitar 170 MB | sekitar 30 MB |
| Sumber | [cs.toronto.edu/~kriz/cifar.html](https://www.cs.toronto.edu/~kriz/cifar.html) | [github.com/zalandoresearch/fashion-mnist](https://github.com/zalandoresearch/fashion-mnist) |

Nilai piksel diskalakan dari 0–255 ke 0–1. Setiap citra diberi sumbu channel eksplisit, `(N, H, W, C)`: CIFAR-10 sudah punya 3 channel (RGB), sedangkan Fashion-MNIST perlu ditambah sumbu channel berukuran 1.

</details>

In [ ]:
DATASETS = {
    "cifar10": {"title": "CIFAR-10", "loader": keras.datasets.cifar10,
                "classes": ["airplane", "automobile", "bird", "cat", "deer",
                            "dog", "frog", "horse", "ship", "truck"]},
    "fashion": {"title": "Fashion-MNIST", "loader": keras.datasets.fashion_mnist,
                "classes": ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
                            "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]},
}
info = DATASETS[DATASET]
CLASSES, TITLE = info["classes"], info["title"]

(x_train, y_train), (x_test, y_test) = info["loader"].load_data()
y_train, y_test = np.asarray(y_train).ravel(), np.asarray(y_test).ravel()
if x_train.ndim == 3:                       # grayscale: add the channel axis
    x_train, x_test = x_train[..., None], x_test[..., None]
x_train = (x_train / 255.0).astype("float32")
x_test = (x_test / 255.0).astype("float32")
if QUICK_RUN:
    x_train, y_train = x_train[:5000], y_train[:5000]

INPUT_SHAPE = x_train.shape[1:]
print(f"{TITLE}: train {x_train.shape}, test {x_test.shape}, input shape {INPUT_SHAPE}")

### Explore the data

A grid of training images and the number of images per class. Both datasets are balanced across classes, so accuracy is a fair summary metric.

<details>
<summary>Bahasa Indonesia</summary>

### Eksplorasi data

Contoh citra latih dan jumlah citra per kelas. Kedua dataset seimbang antar kelas, jadi akurasi adalah ringkasan yang adil.

</details>

In [ ]:
def show(img):
    return img[..., 0] if img.shape[-1] == 1 else img

plt.figure(figsize=(8, 8))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.imshow(show(x_train[i]), cmap="gray")
    plt.title(CLASSES[y_train[i]], fontsize=8)
    plt.axis("off")
plt.suptitle(f"25 {TITLE} training images")
plt.tight_layout()
plt.show()

print("Images per class:", dict(zip(CLASSES, np.bincount(y_train))))
print("Pixel range:", x_train.min(), "to", x_train.max())

## 3. Models

Two models with the same output layer:

- **MLP**: flattens each image into one long vector (3,072 numbers for CIFAR-10, 784 for Fashion-MNIST) and passes it through two dense layers. Every pixel is an independent input, so the model ignores where pixels sit relative to each other.
- **CNN**: two convolution + max-pooling blocks, then a dense layer with dropout. Each 3 × 3 kernel is shared across the whole image (*weight sharing*), so the network learns local patterns such as edges with few parameters.

The architecture is identical for both datasets; only the input shape changes. The builders are functions so the optimizer experiment can create a fresh, identical CNN for every run.

<details>
<summary>Bahasa Indonesia</summary>

## 3. Model

Dua model dengan layer output yang sama:

- **MLP**: meratakan setiap citra menjadi satu vektor panjang (3.072 angka untuk CIFAR-10, 784 untuk Fashion-MNIST) lalu melewatkannya ke dua layer dense. Setiap piksel adalah input yang berdiri sendiri, jadi model tidak memperhatikan posisi piksel terhadap piksel lain.
- **CNN**: dua blok konvolusi + max pooling, lalu layer dense dengan dropout. Setiap kernel 3 × 3 dipakai bersama di seluruh citra (*weight sharing*), sehingga jaringan mempelajari pola lokal seperti tepi dengan sedikit parameter.

Arsitekturnya sama untuk kedua dataset; yang berubah hanya bentuk input. Model dibuat lewat fungsi supaya eksperimen optimizer bisa membuat CNN baru yang identik untuk setiap run.

</details>

In [ ]:
def build_mlp(input_shape=INPUT_SHAPE):
    return keras.Sequential([
        layers.Input(input_shape),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ], name="mlp")


def build_cnn(input_shape=INPUT_SHAPE):
    return keras.Sequential([
        layers.Input(input_shape),
        layers.Conv2D(32, 3, activation="relu", name="conv1"),
        layers.MaxPooling2D(2, name="pool1"),
        layers.Conv2D(64, 3, activation="relu", name="conv2"),
        layers.MaxPooling2D(2, name="pool2"),
        layers.Flatten(name="flatten"),
        layers.Dense(128, activation="relu", name="dense"),
        layers.Dropout(0.3, name="dropout"),
        layers.Dense(10, activation="softmax", name="output"),
    ], name="cnn")


cnn = build_cnn()
cnn.summary()

### Check the parameter count by hand

A convolution layer has `(K × K × C_in + 1) × C_out` parameters: one weight per kernel cell per input channel, plus one bias per filter. A 3 × 3 convolution without padding shrinks the side by 2 (`W − 3 + 1`), and 2 × 2 pooling halves it, rounding down:

| | CIFAR-10 | Fashion-MNIST |
|---|---|---|
| Side length | 32 → 30 → 15 → 13 → 6 | 28 → 26 → 13 → 11 → 5 |
| Flatten | 6 × 6 × 64 = 2,304 | 5 × 5 × 64 = 1,600 |
| conv1 | (3·3·3 + 1) × 32 = 896 | (3·3·1 + 1) × 32 = 320 |
| dense | 2,304 × 128 + 128 = 295,040 | 1,600 × 128 + 128 = 204,928 |

The cell below computes these from the input shape and compares them with what Keras reports.

<details>
<summary>Bahasa Indonesia</summary>

### Cek jumlah parameter secara manual

Layer konvolusi punya `(K × K × C_in + 1) × C_out` parameter: satu bobot untuk setiap sel kernel per channel input, ditambah satu bias per filter. Konvolusi 3 × 3 tanpa padding mengurangi sisi sebanyak 2 (`W − 3 + 1`), dan pooling 2 × 2 membaginya dua (dibulatkan ke bawah):

| | CIFAR-10 | Fashion-MNIST |
|---|---|---|
| Panjang sisi | 32 → 30 → 15 → 13 → 6 | 28 → 26 → 13 → 11 → 5 |
| Flatten | 6 × 6 × 64 = 2.304 | 5 × 5 × 64 = 1.600 |
| conv1 | (3·3·3 + 1) × 32 = 896 | (3·3·1 + 1) × 32 = 320 |
| dense | 2.304 × 128 + 128 = 295.040 | 1.600 × 128 + 128 = 204.928 |

Sel di bawah menghitung angka-angka ini dari bentuk input lalu membandingkannya dengan angka dari Keras.

</details>

In [ ]:
H, _, C = INPUT_SHAPE
side = (H - 2) // 2          # after conv1 and pool1
side = (side - 2) // 2       # after conv2 and pool2
manual = {
    "conv1":   (3 * 3 * C + 1) * 32,
    "pool1":   0,
    "conv2":   (3 * 3 * 32 + 1) * 64,
    "pool2":   0,
    "flatten": 0,
    "dense":   side * side * 64 * 128 + 128,
    "output":  128 * 10 + 10,
}
for name, expected in manual.items():
    layer = cnn.get_layer(name)
    actual = layer.count_params()
    status = "ok" if expected == actual else f"MISMATCH, Keras says {actual:,}"
    print(f"{name:8s} output {str(tuple(layer.output.shape)):18s} manual {expected:>8,}  {status}")
print(f"\nTotal CNN parameters: {cnn.count_params():,}")
print(f"Total MLP parameters: {build_mlp().count_params():,}")

## 4. Train the MLP and the CNN

Both models use Adam and sparse categorical cross-entropy (labels are integers 0–9), and keep the last 10% of the training data for validation. `results` collects the test accuracy, loss, parameter count, and training time of every run, and `histories` keeps the per-epoch curves.

<details>
<summary>Bahasa Indonesia</summary>

## 4. Melatih MLP dan CNN

Kedua model memakai Adam dan sparse categorical cross-entropy (labelnya bilangan bulat 0–9), serta menyisihkan 10% terakhir data latih untuk validasi. `results` mengumpulkan akurasi uji, loss, jumlah parameter, dan waktu latih setiap run, dan `histories` menyimpan kurva per epoch.

</details>

In [ ]:
results, histories, trained = {}, {}, {}


def train(name, model, optimizer="adam", verbose=2):
    """Compile, fit, evaluate, and record one run."""
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    t0 = time.time()
    history = model.fit(x_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE,
                        validation_split=0.1, verbose=verbose)
    seconds = time.time() - t0
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    results[name] = {"params": model.count_params(), "test_accuracy": round(test_acc, 4),
                     "test_loss": round(test_loss, 4), "train_seconds": round(seconds, 1)}
    histories[name], trained[name] = history, model
    print(f"{name}: test accuracy {test_acc:.4f}, {seconds:.0f} s")
    return history


keras.utils.set_random_seed(SEED)
train("MLP", build_mlp())
keras.utils.set_random_seed(SEED)
train("CNN (Adam)", build_cnn())
cnn = trained["CNN (Adam)"]

### Learning curves

Loss and accuracy per epoch for training (solid) and validation (dashed). A validation curve that flattens or turns upward while the training curve keeps improving is a sign of overfitting.

<details>
<summary>Bahasa Indonesia</summary>

### Kurva training

Loss dan akurasi per epoch untuk data latih (garis penuh) dan validasi (garis putus-putus). Kurva validasi yang mendatar atau naik kembali sementara kurva latih terus membaik adalah tanda overfitting.

</details>

In [ ]:
def plot_histories(hs):
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    for name, h in hs.items():
        epochs = range(1, len(h.history["loss"]) + 1)
        line, = ax[0].plot(epochs, h.history["loss"], marker="o", ms=3, label=f"{name} train")
        ax[0].plot(epochs, h.history["val_loss"], "--", color=line.get_color(), label=f"{name} val")
        ax[1].plot(epochs, h.history["accuracy"], marker="o", ms=3, color=line.get_color(), label=f"{name} train")
        ax[1].plot(epochs, h.history["val_accuracy"], "--", color=line.get_color(), label=f"{name} val")
    ax[0].set_title("Loss"); ax[1].set_title("Accuracy")
    for a in ax:
        a.set_xlabel("Epoch"); a.grid(alpha=0.3); a.legend(fontsize=8)
    plt.show()


plot_histories({"MLP": histories["MLP"], "CNN": histories["CNN (Adam)"]})

## 5. Evaluate the CNN

The confusion matrix counts, for every true class (row), how often the CNN predicted each class (column). The diagonal holds correct predictions. The cell after it lists the class pairs the model confuses most and the accuracy of each class.

<details>
<summary>Bahasa Indonesia</summary>

## 5. Evaluasi CNN

Confusion matrix menghitung, untuk setiap kelas asli (baris), seberapa sering CNN memprediksi setiap kelas (kolom). Diagonal berisi prediksi yang benar. Sel berikutnya menampilkan pasangan kelas yang paling sering tertukar dan akurasi tiap kelas.

</details>

In [ ]:
prob = cnn.predict(x_test, verbose=0)
y_pred = prob.argmax(axis=1)
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(9, 9))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
ax.set_title(f"CNN confusion matrix, {TITLE} test set")
plt.show()

In [ ]:
off_diag = cm.copy()
np.fill_diagonal(off_diag, 0)
print("Most confused pairs (true -> predicted):")
for k in np.argsort(off_diag, axis=None)[::-1][:5]:
    i, j = np.unravel_index(k, cm.shape)
    print(f"  {CLASSES[i]:12s} -> {CLASSES[j]:12s} {off_diag[i, j]} images")

print("\nAccuracy per class:")
for i, name in enumerate(CLASSES):
    print(f"  {name:12s} {cm[i, i] / cm[i].sum():.3f}")

### Misclassified images

The first 15 test images the CNN got wrong, with the true label, the prediction, and the model's confidence (the highest softmax probability). At this resolution many are hard even for a person: on CIFAR-10, animals in unusual poses or cars that look like trucks; on Fashion-MNIST, shirts, pullovers, and coats that share the same silhouette.

<details>
<summary>Bahasa Indonesia</summary>

### Citra yang salah klasifikasi

15 citra uji pertama yang salah ditebak CNN, beserta label asli, prediksi, dan keyakinan model (probabilitas softmax tertinggi). Pada resolusi ini banyak yang sulit dikenali manusia sekalipun: di CIFAR-10, hewan dengan pose tidak biasa atau mobil yang mirip truk; di Fashion-MNIST, kemeja, pullover, dan mantel yang siluetnya sama.

</details>

In [ ]:
wrong = np.where(y_pred != y_test)[0]
print(f"Misclassified: {len(wrong)} of {len(y_test)}")

plt.figure(figsize=(12, 7))
for i, idx in enumerate(wrong[:15]):
    plt.subplot(3, 5, i + 1)
    plt.imshow(show(x_test[idx]), cmap="gray")
    plt.title(f"true: {CLASSES[y_test[idx]]}\npred: {CLASSES[y_pred[idx]]}\n"
              f"confidence {prob[idx].max():.0%}", fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 6. Optimizer experiment

The same CNN is trained again with eight optimizers. Architecture, seed (so the starting weights are identical), epochs, batch size, and validation split stay fixed, so any difference comes from the optimizer.

| Optimizer | How it updates the weights | When to use it |
|---|---|---|
| SGD | Plain step against the gradient | Learning the basics, building a baseline |
| RMSprop | Scales the learning rate by a running average of squared gradients | An alternative for training neural networks |
| Adam | Momentum plus adaptive per-parameter scaling | A practical first choice for CNNs |
| AdamW | Adam with decoupled weight decay | When you want weight regularization |
| Adagrad | Step size shrinks with accumulated squared gradients | Sparse features or rare gradients |
| Nadam | Adam with Nesterov-style momentum | An Adam alternative for experiments |
| Adadelta | Adagrad variant that limits the accumulation | Experiments with adaptive methods |
| SGD + momentum | SGD that keeps part of the previous update (lr 0.05, momentum 0.9, Nesterov) | Extra run: can SGD catch up with momentum? |

All optimizers except the last use Keras default learning rates. SGD (0.01, no momentum), Adagrad, and Adadelta (0.001) learn slowly with these defaults; that is part of what the experiment shows.

<details>
<summary>Bahasa Indonesia</summary>

## 6. Eksperimen optimizer

CNN yang sama dilatih ulang dengan delapan optimizer. Arsitektur, seed (sehingga bobot awalnya identik), epoch, batch size, dan pembagian validasi dibuat tetap, jadi perbedaan hasil hanya berasal dari optimizer.

| Optimizer | Karakteristik | Kapan digunakan? |
|---|---|---|
| SGD | Pembaruan sederhana berdasarkan gradien | Memahami dasar optimisasi dan membuat baseline |
| RMSprop | Menyesuaikan learning rate berdasarkan riwayat kuadrat gradien | Alternatif untuk pelatihan jaringan saraf |
| Adam | Menggabungkan momentum dan penskalaan adaptif dari gradien | Pilihan awal yang praktis untuk CNN |
| AdamW | Adam dengan decoupled weight decay | Ketika ingin menambahkan regularisasi bobot |
| Adagrad | Menyesuaikan langkah berdasarkan akumulasi kuadrat gradien | Masalah dengan fitur atau gradien yang jarang muncul |
| Nadam | Adam dengan momentum bergaya Nesterov | Alternatif Adam untuk eksperimen |
| Adadelta | Metode adaptif yang membatasi akumulasi gradien Adagrad | Eksperimen optimisasi adaptif |
| SGD + momentum | SGD yang membawa sebagian pembaruan sebelumnya (lr 0,05, momentum 0,9, Nesterov) | Run tambahan: apakah SGD bisa menyusul dengan momentum? |

Semua optimizer kecuali yang terakhir memakai learning rate bawaan Keras. SGD (0,01, tanpa momentum), Adagrad, dan Adadelta (0,001) belajar lambat dengan nilai bawaan ini; itu bagian dari temuan eksperimen.

</details>

In [ ]:
OPTIMIZERS = {
    "SGD":          lambda: keras.optimizers.SGD(),
    "RMSprop":      lambda: keras.optimizers.RMSprop(),
    "Adam":         lambda: keras.optimizers.Adam(),
    "AdamW":        lambda: keras.optimizers.AdamW(),
    "Adagrad":      lambda: keras.optimizers.Adagrad(),
    "Nadam":        lambda: keras.optimizers.Nadam(),
    "Adadelta":     lambda: keras.optimizers.Adadelta(),
    "SGD momentum": lambda: keras.optimizers.SGD(learning_rate=0.05, momentum=0.9, nesterov=True),
}

rows = []
for name, make_optimizer in OPTIMIZERS.items():
    keras.utils.set_random_seed(SEED)
    h = train(name, build_cnn(), optimizer=make_optimizer(), verbose=0)
    val = h.history["val_accuracy"]
    rows.append({"optimizer": name, "test accuracy": results[name]["test_accuracy"],
                 "test loss": results[name]["test_loss"], "best val accuracy": max(val),
                 "best epoch": int(np.argmax(val)) + 1,
                 "train-val gap": h.history["accuracy"][-1] - val[-1],
                 "seconds": results[name]["train_seconds"]})

df_opt = pd.DataFrame(rows).set_index("optimizer").sort_values("test accuracy", ascending=False)
display(df_opt.round(4))

### Compare the optimizers

Validation curves for all eight runs, then test accuracy as a bar chart with the MLP as a dashed reference line.

<details>
<summary>Bahasa Indonesia</summary>

### Membandingkan optimizer

Kurva validasi untuk kedelapan run, lalu akurasi uji dalam diagram batang dengan MLP sebagai garis pembanding putus-putus.

</details>

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))
for name in OPTIMIZERS:
    h = histories[name]
    epochs = range(1, len(h.history["loss"]) + 1)
    ax[0].plot(epochs, h.history["val_loss"], marker="o", ms=3, label=name)
    ax[1].plot(epochs, h.history["val_accuracy"], marker="o", ms=3, label=name)
ax[0].set_title("Validation loss"); ax[1].set_title("Validation accuracy")
for a in ax:
    a.set_xlabel("Epoch"); a.grid(alpha=0.3); a.legend(fontsize=8)
plt.show()

order = df_opt["test accuracy"].sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(order.index, order.values, color="#2563EB")
ax.axvline(results["MLP"]["test_accuracy"], ls="--", color="grey", label="MLP (Adam)")
for i, v in enumerate(order.values):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=9)
ax.set_xlim(0, order.max() + 0.08)
ax.set_xlabel("Test accuracy")
ax.set_title(f"{TITLE}: CNN test accuracy per optimizer ({EPOCHS} epochs)")
ax.legend(loc="lower right")
plt.show()

In [ ]:
# Train vs validation for the three best optimizers, to look for overfitting
plot_histories({name: histories[name] for name in df_opt.index[:3]})

### What the results show

From the runs published in the dashboard (your numbers will differ slightly):

| | CIFAR-10 | Fashion-MNIST |
|---|---|---|
| Best CNN | 68.9% (Nadam) | RMSprop at 91.0% |
| CNN with Adam | 67.9% | 91.0% |
| MLP with Adam | 46.9% | 87.3% |
| SGD / Adagrad / Adadelta | 47.5% / 31.9% / 16.1% | 81.6% / 73.8% / 50.4% |

- The adaptive optimizers (Adam, AdamW, Nadam, RMSprop) end close together on both datasets. With one seed, gaps of about a point are not enough to call one of them better.
- SGD, Adagrad, and Adadelta fall behind because their default learning rates are too small for 10 epochs. The gap is much larger on CIFAR-10, the harder dataset.
- The CNN's advantage over the MLP is far larger on CIFAR-10 than on Fashion-MNIST: color photos with varied backgrounds need local, position-independent features that an MLP cannot learn efficiently, while centered clothing silhouettes on a black background are already easy for an MLP.

<details>
<summary>Bahasa Indonesia</summary>

### Apa yang ditunjukkan hasilnya

Dari run yang dipublikasikan di dashboard (angka di run-mu akan sedikit berbeda):

| | CIFAR-10 | Fashion-MNIST |
|---|---|---|
| CNN terbaik | 68,9% (Nadam) | RMSprop, 91,0% |
| CNN dengan Adam | 67,9% | 91,0% |
| MLP dengan Adam | 46,9% | 87,3% |
| SGD / Adagrad / Adadelta | 47,5% / 31,9% / 16,1% | 81,6% / 73,8% / 50,4% |

- Optimizer adaptif (Adam, AdamW, Nadam, RMSprop) berakhir berdekatan di kedua dataset. Dengan satu seed, selisih sekitar satu poin belum cukup untuk menyebut salah satunya lebih baik.
- SGD, Adagrad, dan Adadelta tertinggal karena learning rate bawaannya terlalu kecil untuk 10 epoch. Ketertinggalannya jauh lebih besar di CIFAR-10, dataset yang lebih sulit.
- Keunggulan CNN atas MLP jauh lebih besar di CIFAR-10 daripada di Fashion-MNIST: foto berwarna dengan latar beragam membutuhkan fitur lokal yang tidak bergantung posisi, dan itu tidak bisa dipelajari MLP secara efisien, sedangkan siluet pakaian di tengah latar hitam sudah mudah bagi MLP.

</details>

## 7. Save models and results

Saves every trained model to `models/<dataset>/` (`mlp.keras` and `cnn-<optimizer>.keras`, each with a `.history.json`) and the summary table to `results/<dataset>_results.csv`. This is the same layout `src/train.py` writes, so `src/export_dashboard.py` can turn a Colab run into dashboard data. In Colab, download the files from the file panel or copy them to Google Drive.

<details>
<summary>Bahasa Indonesia</summary>

## 7. Menyimpan model dan hasil

Menyimpan setiap model ke `models/<dataset>/` (`mlp.keras` dan `cnn-<optimizer>.keras`, masing-masing dengan `.history.json`) dan tabel ringkasan ke `results/<dataset>_results.csv`. Susunannya sama dengan yang ditulis `src/train.py`, jadi `src/export_dashboard.py` bisa mengubah hasil run di Colab menjadi data dashboard. Di Colab, unduh berkasnya lewat panel file atau salin ke Google Drive.

</details>

In [ ]:
models_dir, results_dir = Path("models") / DATASET, Path("results")
models_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(exist_ok=True)

def slug(name):
    return name.lower().replace(" ", "-")

for name, model in trained.items():
    if name == "CNN (Adam)":
        continue                          # same run as "Adam" in the optimizer loop
    file = "mlp" if name == "MLP" else f"cnn-{slug(name)}"
    model.save(models_dir / f"{file}.keras")
    r = results[name]
    json.dump({"name": name, "params": r["params"], "time": r["train_seconds"],
               "test_accuracy": r["test_accuracy"], "test_loss": r["test_loss"],
               "hist": {k: [round(float(v), 4) for v in vs] for k, vs in histories[name].history.items()}},
              open(models_dir / f"{file}.history.json", "w"), indent=1)

summary = pd.DataFrame(results).T.astype({"params": int})
summary.to_csv(results_dir / f"{DATASET}_results.csv", index_label="run")
display(summary)
print("Saved to", models_dir.resolve())

## Next: the other dataset

Change `DATASET` in the first code cell to `"fashion"` (or back to `"cifar10"`) and run all cells again. Both runs save to separate folders, so nothing is overwritten.

<details>
<summary>Bahasa Indonesia</summary>

## Berikutnya: dataset lainnya

Ganti `DATASET` di sel kode pertama menjadi `"fashion"` (atau kembali ke `"cifar10"`) lalu jalankan semua sel lagi. Kedua run disimpan di folder terpisah, jadi tidak ada yang tertimpa.

</details>